# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

import os
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult, EvaluationReport, PydanticOutputParser

In [3]:
# TODO: Load environment variables
import os
load_dotenv(dotenv_path="config.env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY)
print(OPENAI_API_KEY)
print(os.getenv("OPENAI_BASE_URL"))

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
voc-71328759916886552416536a5a3865718728.78066438
https://openai.vocareum.com/v1


In [4]:
vector_store_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
vector_store = vector_store_manager.get_or_create_store("udaplay")


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

def retrieve_game(query: str, vector_store, n_results: int = 3):
    """
    Semantic search: Finds most relevant results in the vector DB.

    Args:
        query: a question about the game industry.

    Returns:
        A list of matching game records.
    """
    results = vector_store._collection.query(
        query_texts=[query],
        n_results=n_results
    )

    matches = results.get("metadatas", [[]])[0]
    return matches if matches else []

    
@tool
def get_games(query: str):
    """
    Retrieve relevant game information from the internal vector database.
    """
    return retrieve_game(query, vector_store)



In [6]:
print(get_games("games published by Sony Computer Entertainment"))


[{'Publisher': 'Sony Interactive Entertainment', 'Name': "Marvel's Spider-Man", 'YearOfRelease': 2018, 'Genre': 'Action-adventure', 'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.', 'Platform': 'PlayStation 4'}, {'Platform': 'PlayStation 3', 'Genre': 'Racing', 'YearOfRelease': 2010, 'Publisher': 'Sony Computer Entertainment', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', 'Name': 'Gran Turismo 5'}, {'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Publisher': 'Sony Computer Entertainment', 'YearOfRelease': 1997, 'Name': 'Gran Turismo', 'Genre': 'Racing', 'Platform': 'PlayStation 1'}]


#### Evaluate Retrieval Tool

In [7]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result


def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Based on the user's question and on the list of retrieved documents,
    it analyzes the usability of the documents to respond to that question.

    Args:
        question: original question from user
        retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    Returns:
        EvaluationReport:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    llm_judge = LLM(model="gpt-4o-mini")

    judge_prompt = f"""
    Your task is to evaluate if the retrieved documents are enough to respond to the user's question.

    User question:
    {question}

    Retrieved documents:
    {retrieved_docs}

    Decide whether these documents are useful for answering the question.

    Rules:
    - Mark useful as true only if the documents contain enough relevant information to answer the question.
    - Mark useful as false if the documents are missing key information, are off-topic, or are too weak to support a good answer.
    - Give a detailed explanation, so it's possible to take an action to accept it or not.

    Return:
    - useful: true or false
    - description: explanation
    """

    judge_response = llm_judge.invoke(
        input=judge_prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)

    try:
        report = parser.parse(judge_response)
    except Exception as e:
        print(f"Parsing error: {e}")
        print(f"Judge response: {judge_response}")

        report = EvaluationReport(
            useful=len(retrieved_docs) > 0,
            description=f"Fallback evaluation due to parsing error: {str(e)}"
        )

    return report


@tool
def judge_retrieval(question: str, retrieved_docs):

    return evaluate_retrieval(question, retrieved_docs)


In [8]:
docs = retrieve_game("games published by Sony Computer Entertainment", vector_store)
report = judge_retrieval(
    "games published by Sony Computer Entertainment",
    docs
)

print(report)
print(report.useful)
print(report.description)


useful=True description="The retrieved documents provide relevant information about games published by Sony Computer Entertainment, which directly addresses the user's question. The documents include specific titles such as 'Gran Turismo 5' and 'Gran Turismo' along with their release years, genres, and descriptions, clearly indicating that they are published by Sony Computer Entertainment. Additionally, the inclusion of 'Marvel's Spider-Man' under 'Sony Interactive Entertainment' is also relevant, as it is a subsidiary of Sony Computer Entertainment. Overall, the documents collectively offer a sufficient overview of games published by the company, making them useful for answering the user's inquiry."
True
The retrieved documents provide relevant information about games published by Sony Computer Entertainment, which directly addresses the user's question. The documents include specific titles such as 'Gran Turismo 5' and 'Gran Turismo' along with their release years, genres, and descri

#### Game Web Search Tool

In [9]:

from typing import Dict
import os
from datetime import datetime
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [10]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

def web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API.

    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced'
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }

    return formatted_results

@tool
def game_web_search(question: str) -> Dict:
    """
    Web search tool for game industry questions.

    args:
    - question: a question about game industry.
    """
    return web_search(question)

In [11]:
result = game_web_search("What are recent trends in the video game industry?")
print(result["answer"])
print(result["results"][:2])


Recent trends in the video game industry include consolidation among top companies, growth in mobile gaming, and the rise of cloud gaming. Generative AI and user-generated content are also driving innovation.
[{'url': 'https://www.juegostudio.com/blog/recent-trends-reforming-gaming-industry', 'title': 'Top Video Game Industry Trends for 2026 | Juego Studios', 'content': 'Key policy shifts:\n\n AI-generated asset labeling becomes legally required\n Loot box restrictions expand across Europe and emerging markets\n Fair-play certifications increase visibility in app stores\n Wellbeing tools—like fatigue detection or break prompts—become standard for live-service titles\n\nThese guardrails protect players while helping studios build sustainable long-term communities.\n\n## Conclusion\n\nThe most significant video game industry trends for 2026 revolve around intelligent automation, accessible design, alignment with global audiences, and seamless multi-device play. As AI accelerates creation

### Agent

In [12]:
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult, EvaluationReport, PydanticOutputParser

In [13]:
from lib.agentic_rag import AgenticRAGState, AgenticRAG

In [14]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [15]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?


# Create the agent
agent = AgenticRAG(llm=rag_llm, vector_store=vector_store, web_search_tool=game_web_search,)

In [16]:
# Question 1
run1 = agent.invoke("When Pokémon Gold and Silver was released?")
final_state = run1.get_final_state()
print(final_state.get("evaluation_description"))
print(final_state.get("answer"))


[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate_retrieval
WEB SEARCH CHECK - evaluation_useful: True
[StateMachine] Executing step: web_search
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
{"useful":true,"description":"The retrieved records include the game 'Pokémon Gold and Silver' along with its release year, 1999, which directly answers the user's question about when the game was released."}
Pokémon Gold and Silver was released in 1999.


In [17]:
# Question 2
run2 = agent.invoke("Which one was the first 3D platformer Mario game?")
final_state = run2.get_final_state()
print(final_state.get("evaluation_description"))
print(final_state.get("answer"))


[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate_retrieval
WEB SEARCH CHECK - evaluation_useful: True
[StateMachine] Executing step: web_search
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
{"useful":true,"description":"The retrieved records include 'Super Mario 64', which is identified as a 3D platformer released in 1996. This directly answers the user's question about the first 3D platformer Mario game."}
The first 3D platformer Mario game is "Super Mario 64," released in 1996 for the Nintendo 64.


In [18]:
# Question 3
run3 = agent.invoke("Was Mortal Kombat X released for Playstation 5?")
final_state = run3.get_final_state()
print(final_state.get("evaluation_description"))
print(final_state.get("answer"))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate_retrieval
WEB SEARCH CHECK - evaluation_useful: False
[StateMachine] Executing step: web_search
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
{"useful":false,"description":"The retrieved records do not contain any information about Mortal Kombat X or its availability on PlayStation 5. They only include details about other games."}
Mortal Kombat X was released for PlayStation 4, not PlayStation 5.


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes